# Lab 3 — 에이전트 루프

> **이론 복습 — Session 3 슬라이드**
> - ReAct 루프: 추론(Reason) → 행동(Act) → 관찰(Observe) → 반복
> - 종료 조건: 모델이 도구를 더 안 부르면 멈춘다 / `MAX_STEPS` 안전망
> - 메모리 = `messages` 리스트. LLM은 stateless, 기억은 우리가 운반한다.

## 학습 목표
1. Lab 2의 "한 바퀴"를 *반복하는* `Agent` 클래스를 완성한다
2. 도구가 **연쇄 호출**되는 멀티스텝 질문을 푼다
3. DTUMOS 스타일 **미니 시뮬레이터**를 도구로 추가한다


## 0. 준비
`labs/` 폴더에서 실행하세요. 키가 없으면 `MockLLM` 으로 동작합니다
(이 경우 에이전트는 도구를 한 번만 부르고 끝냅니다 — 메커니즘은 동일합니다).


In [ ]:
import json
from common.llm import LLMClient
from common import tools, mini_sim

## 1. 복습 — 메모리는 `messages` 리스트

에이전트의 기억은 메시지 딕셔너리의 리스트입니다. 세 가지 역할(role)이 있습니다.

```
{"role": "user",      "content": "..."}                  사용자 질문
{"role": "assistant", "content": "...", "tool_calls": [...]}  모델의 응답
{"role": "tool",      "name": "...",   "content": "..."}  도구 실행 결과
```

매 스텝 모델은 이 리스트 **전체**를 다시 보고 다음 행동을 정합니다.


## 2. `Agent` 클래스 — 🔧 이 셀에 TODO가 있습니다

아래 `Agent` 의 `run()` 은 Session 3 슬라이드의 의사코드와 똑같습니다.
한 곳, `_finalize()` 메서드만 비어 있습니다 — TODO 주석을 따라 완성하세요.


In [ ]:
class Agent:
    """An LLM agent that loops: reason -> act -> observe."""

    def __init__(self, llm, schemas, functions, system=None, max_steps=6):
        self.llm = llm
        self.schemas = schemas        # tool schemas the model sees
        self.functions = functions    # {name: callable} that actually run
        self.system = system
        self.max_steps = max_steps

    def _run_tool(self, name, args):
        """Run one tool. Errors are returned, not raised."""
        if name not in self.functions:
            return {"error": f"unknown tool: {name}"}
        try:
            return self.functions[name](**(args or {}))
        except Exception as exc:
            return {"error": f"{type(exc).__name__}: {exc}"}

    def _finalize(self, messages):
        # 🔧 TODO: the loop hit max_steps without finishing. Instead of giving
        # up, ask the LLM ONE more time WITHOUT tools so it answers from what
        # it already has. Hint: self.llm.generate(messages, self.system)
        # returns an LLMResponse; its .text is the answer.
        return "(최대 스텝에 도달했습니다.)"   # <-- TODO: replace this line

    def run(self, question, verbose=True):
        """Answer `question` by looping reason -> act -> observe."""
        messages = [{"role": "user", "content": question}]

        for step in range(self.max_steps):
            reply = self.llm.generate_with_tools(messages, self.schemas, self.system)

            if not reply.wants_tool:               # no tool wanted -> done
                return reply.text

            messages.append({"role": "assistant", "content": reply.text,
                             "tool_calls": reply.tool_calls})
            for call in reply.tool_calls:
                result = self._run_tool(call.name, call.args)
                if verbose:
                    print(f"  step {step + 1}: {call.name}({call.args})")
                messages.append({"role": "tool", "name": call.name,
                                 "content": json.dumps(result, ensure_ascii=False,
                                                        default=str)})
        return self._finalize(messages)

print("Agent class defined.")

## 3. 단일 스텝 질문

먼저 도구 한 번이면 풀리는 질문입니다. `step 1: ...` 출력으로 행동을 관찰하세요.


In [ ]:
analysis_tools = tools.get_schemas()      # all 5 mobility tool schemas
analysis_funcs = dict(tools.TOOLBOX)      # {name: function}

agent = Agent(
    llm=LLMClient(),
    schemas=analysis_tools,
    functions=analysis_funcs,
    system="You are a mobility analyst. Answer in Korean, concisely.",
)
print(agent.run("통근 통행이 가장 많은 3개 구간을 알려줘."))

## 4. 멀티스텝 질문

이제 도구가 **두 번 이상** 필요한 질문입니다. 실제 키로 돌리면 도구가 연쇄
호출되는 것을, `step 1`, `step 2` ... 출력으로 볼 수 있습니다.


In [ ]:
answer = agent.run(
    "자족도가 가장 낮은 생활권을 찾고, "
    "그 생활권과 '서울 생활권' 사이의 통근 통행량을 알려줘."
)
print()
print("답변:", answer)

## 5. 시뮬레이션 도구 추가 — DTUMOS 스타일

지금까지는 데이터 *조회* 도구뿐이었습니다. 이번엔 `common/mini_sim.py` 의
미니 택시 시뮬레이터를 도구로 추가합니다. 같은 에이전트가 분석과 시뮬레이션을
모두 할 수 있게 됩니다.


In [ ]:
SIM_SCHEMA = {
    "name": "run_mini_simulation",
    "description": "Run a taxi-dispatch simulation and return service metrics "
                   "(service rate, average wait time).",
    "parameters": {
        "type": "object",
        "properties": {
            "fleet_size": {"type": "integer",
                           "description": "Number of taxis (1-60)."},
            "dispatch": {"type": "string", "enum": ["nearest", "fifo"],
                         "description": "Dispatch strategy."},
        },
        "required": [],
    },
}

sim_agent = Agent(
    llm=LLMClient(),
    schemas=analysis_tools + [SIM_SCHEMA],
    functions={**analysis_funcs, "run_mini_simulation": mini_sim.run_mini_simulation},
    system="You are a mobility analyst. Use tools, then answer in Korean.",
)
print(sim_agent.run("택시 10대일 때와 40대일 때 평균 대기시간을 비교해줘."))

## 정리

- `Agent` = Lab 2의 한 바퀴를 `for step in range(MAX_STEPS)` 로 **반복**
- 종료 조건(`not wants_tool`) + 안전망(`_finalize`) 으로 루프는 항상 끝난다
- 도구를 더하면(`SIM_SCHEMA`) 에이전트의 능력이 곧바로 늘어난다

> 이 `Agent` 클래스의 완성본은 `common/agent.py` 에 있습니다 —
> 다음 실습(Lab 4, 5)은 거기서 `from common.agent import Agent` 로 가져다 씁니다.

**다음 — Session 4**: 에이전트를 여러 개로 나누고(멀티에이전트), 지식을 주고(RAG),
표준화하고(MCP), 테스트하는 법.
